In [ ]:
# 필요한 lib 목록들
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import os
from time import sleep
import csv
import datetime
import time
import pytz

from pathlib import Path

# 노트북을 어느 하위 폴더에서 실행해도 저장소 루트를 찾습니다.
_current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (_current_dir, *_current_dir.parents)
    if (path / "notebooks").is_dir() and (path / "data").is_dir()
)
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"


In [ ]:
# 환경변수에서 API 키를 불러오고 기본 URL 설정
api_key = os.environ["TONCENTER_API_KEY"]
base_url = "https://toncenter.com/api/v3/transactions"
headers = {
    "X-API-Key": api_key
}

output_dir = DATA_DIR / 'raw' / 'ton'
output_dir.mkdir(parents=True, exist_ok=True)

# 시작 및 종료 날짜 설정
start_date = datetime.datetime(2024, 1, 1, tzinfo=pytz.timezone('UTC'))
end_date = datetime.datetime(2024, 1, 2, tzinfo=pytz.timezone('UTC'))

# CSV 파일에 저장할 필드 이름 설정
fieldnames = [
    'timestamp', 'block_height', 'master_block_height', 'sender_address', 
    'receiver_address', 'amount', 'trx_hash', 'trx_gasPrice'
]


# 하루 단위로 데이터를 가져오기 위해 datetime.timedelta 사용
one_day = datetime.timedelta(days=1)
current_date = start_date

while current_date < end_date:
    next_date = current_date + one_day
    
    start_timestamp = int(current_date.timestamp())
    end_timestamp = int(next_date.timestamp())
    
    # CSV 파일 경로 설정
    output_csv = output_dir / f'TONCOIN_{current_date.strftime("%Y-%m-%d")}.csv'
    
    limit = 256
    offset = 0
    page_number = 1
    
    # CSV 파일 생성 및 필드 이름 작성
    with output_csv.open('w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        while True:
            url = f"{base_url}?start_utime={start_timestamp}&end_utime={end_timestamp}&limit={limit}&offset={offset}&sort=asc"
            try:
                response = requests.get(url, headers=headers, timeout=10)  # 타임아웃 설정 (10초)
                response.raise_for_status()  # HTTP 에러 발생 시 예외 발생
            except requests.exceptions.RequestException as e:
                print(f"Request failed: {e}")
                time.sleep(5)  # 잠시 대기 후 재시도
                continue
            
            data = response.json()
            transfers = data['transactions']
            if not transfers:
                break
            
            # JSON 데이터에서 필요한 필드 추출하여 리스트에 추가
            for transfer in transfers:
                for out_msg in transfer['out_msgs']:
                    row = {
                        'timestamp': datetime.datetime.utcfromtimestamp(transfer['now']).strftime('%Y-%m-%d %H:%M:%S'),
                        'block_height': transfer['block_ref']['seqno'],
                        'master_block_height': transfer['mc_block_seqno'],
                        'sender_address': out_msg['source'],
                        'receiver_address': out_msg['destination'],
                        'amount': out_msg['value'],
                        'trx_hash': transfer['hash'],
                        'trx_gasPrice': transfer['total_fees']
                    }
                    writer.writerow(row)
            
            offset += limit  # 다음 페이지로 이동
            if page_number%50 == 0:
                print(f"Page {page_number} for date {current_date.strftime('%Y-%m-%d')} is complete!")
            page_number += 1

            # 너무 많은 요청을 하지 않기 위해 잠시 대기
            time.sleep(2)
    
    # 현재 날짜에 대한 CSV 파일 저장 완료 메시지 출력
    print(f"Data for {current_date.strftime('%Y-%m-%d')} successfully saved to {output_csv}")
    
    # 다음 날짜로 이동
    current_date = next_date

print(f'Data successfully saved to CSV files in {output_dir}')
